# Orion UI `ui.table` Showcase

This test notebook demonstrates the `orion_ui.table` component with several common DataFrame shapes and configuration options:

- Paginated tables for larger datasets
- Compact tables with hidden index
- Long-text truncation via `max_cell_chars`
- Mixed data types, missing values, and categorical fields
- Multiple table instances on one page

In [13]:
import pandas as pd
import numpy as np
import orion_ui as ui

np.random.seed(42)

# Make notebook display settings friendlier for fallback/static displays.
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

## Sample data

The tables below use synthetic datasets designed to exercise sorting, filtering, pagination, missing values, categorical data, dates, currency-like numeric fields, and long text.

In [14]:
n = 250
regions = ["North", "South", "East", "West", "Central"]
segments = ["Consumer", "Corporate", "Small Business", "Enterprise"]
statuses = ["New", "Qualified", "Proposal", "Won", "Lost"]

sales_df = pd.DataFrame({
    "order_id": [f"ORD-{10000+i}" for i in range(n)],
    "order_date": pd.date_range("2025-01-01", periods=n, freq="D"),
    "region": np.random.choice(regions, n),
    "segment": np.random.choice(segments, n),
    "status": np.random.choice(statuses, n, p=[0.18, 0.24, 0.22, 0.21, 0.15]),
    "units": np.random.randint(1, 25, n),
    "unit_price": np.round(np.random.uniform(15, 450, n), 2),
    "discount": np.round(np.random.choice([0, 0.05, 0.10, 0.15, 0.20], n), 2),
})

sales_df["revenue"] = np.round(sales_df["units"] * sales_df["unit_price"] * (1 - sales_df["discount"]), 2)
sales_df["margin_pct"] = np.round(np.random.uniform(0.08, 0.42, n), 3)
sales_df.loc[np.random.choice(sales_df.index, 12, replace=False), "margin_pct"] = np.nan

sales_df.head()

,order_id,order_date,region,segment,status,units,unit_price,discount,revenue,margin_pct
0,ORD-10000,2025-01-01,West,Consumer,Proposal,10,286.48,0.15,2435.08,0.139
1,ORD-10001,2025-01-02,Central,Enterprise,Won,24,249.80,0.15,5095.92,0.342
2,ORD-10002,2025-01-03,East,Enterprise,New,17,205.85,0.10,3149.50,0.402
3,ORD-10003,2025-01-04,Central,Corporate,Proposal,10,266.21,0.05,2529.00,0.137
4,ORD-10004,2025-01-05,Central,Enterprise,New,17,169.58,0.15,2450.43,0.342


## Basic paginated table

A larger sales-style dataset using the default paginated table mode. The `source` argument records a readable pandas expression for saved table views.

In [15]:
ui.table(
    sales_df,
    source="sales_df",
    mode="paginated",
    page_size=25,
    show_index=False,
)

OrionUI(Table, id='orion-ui-38cc002cf835446cad081d70231f4b42')

## Compact mixed-type table

This smaller table highlights mixed columns: numbers, booleans, datetimes, categories, missing values, and labels.

In [17]:
customers_df = pd.DataFrame({
    "customer": ["Acme Co", "BrightPath", "Canyon Labs", "DeltaWorks", "Evergreen", "Futura", "Graphite", "Helio"] ,
    "tier": pd.Categorical(["Gold", "Silver", "Bronze", "Gold", "Silver", "Enterprise", "Bronze", "Gold"]),
    "active": [True, True, False, True, np.nan, True, False, True],
    "signup_date": pd.to_datetime(["2024-01-14", "2024-03-02", "2024-05-19", "2024-06-22", None, "2024-09-03", "2024-11-18", "2025-01-07"]),
    "account_value": [125000.50, 48800.00, 9100.25, 223400.00, np.nan, 712000.75, 18650.00, 157900.10],
    "health_score": [92, 81, 44, 96, 63, 88, 51, 90],
})

ui.table(
    customers_df,
    source="customers_df",
    page_size=10,
    show_index=True,
)

OrionUI(Table, id='orion-ui-f0bb7274edab4d00bf50b42fdb384417')

## Long text and truncation

Use `max_cell_chars` to keep tables readable when cells contain longer notes or descriptions.

In [18]:
notes_df = pd.DataFrame({
    "ticket_id": ["TCK-001", "TCK-002", "TCK-003", "TCK-004", "TCK-005"],
    "priority": ["High", "Medium", "Low", "High", "Medium"],
    "owner": ["Nora", "Sam", "Iris", "Kai", "Maya"],
    "summary": [
        "Investigate intermittent authentication failures affecting a subset of enterprise SSO users during peak login windows.",
        "Update dashboard copy after product terminology changes and validate screenshots across all supported themes.",
        "Backfill missing usage records from the historical import job and reconcile totals with finance exports.",
        "Resolve duplicate webhook deliveries by adding idempotency checks and improving retry observability.",
        "Document recommended onboarding workflow for customer success managers and include examples for common segments.",
    ],
    "last_updated": pd.to_datetime(["2025-02-03 10:15", "2025-02-05 14:22", "2025-02-07 09:01", "2025-02-07 16:48", "2025-02-08 11:30"]),
})

ui.table(
    notes_df,
    source="notes_df",
    page_size=5,
    show_index=False,
    max_cell_chars=55,
)

OrionUI(Table, id='orion-ui-37e945aa2428434e9bc1c0fd68739aaf')

## Summary table

`ui.table` also works well for pre-aggregated DataFrames.

In [19]:
summary_df = (
    sales_df
    .groupby(["region", "status"], as_index=False)
    .agg(
        orders=("order_id", "count"),
        total_units=("units", "sum"),
        total_revenue=("revenue", "sum"),
        avg_margin_pct=("margin_pct", "mean"),
    )
    .sort_values(["region", "total_revenue"], ascending=[True, False])
)
summary_df["total_revenue"] = summary_df["total_revenue"].round(2)
summary_df["avg_margin_pct"] = summary_df["avg_margin_pct"].round(3)

ui.table(
    summary_df,
    source="summary_df",
    page_size=15,
    show_index=False,
)

OrionUI(Table, id='orion-ui-ed30836ef13a4c989cb8269194c5ca9d')

## Multiple tables in a composed UI

Tables can be placed inside Orion UI layout components such as `ui.grid`, `ui.card`, and `ui.page`.

In [20]:
top_orders_df = sales_df.nlargest(10, "revenue")[["order_id", "region", "segment", "status", "units", "revenue"]]
status_counts_df = sales_df["status"].value_counts().rename_axis("status").reset_index(name="orders")

ui.page(
    ui.grid(
        ui.card(
            ui.table(
                top_orders_df,
                source="top_orders_df",
                page_size=10,
                show_index=False,
            ),
            title="High-value orders",
        ),
        ui.card(
            ui.table(
                status_counts_df,
                source="status_counts_df",
                page_size=10,
                show_index=False,
            ),
            title="Pipeline distribution",
        ),
        columns=2,
    ),
    gap="lg",
    padding="md",
)

OrionUI(Page, id='orion-ui-bb78551547714c54aac1e3dbfe9a0899')